In [1]:
## Separate low-High mtDNA-CN_samples cancer type-wise 

import re 
import statistics
import pandas as pd 


purfilt = {}
data = open('../results/PCAWG_MitoData_collected.txt')
lines = data.readlines()
for line in lines: 
    if not line.startswith('TumorType'):
        elems = line.split() 
        #print(elems[3],elems[28])
        if elems[28] != 'NA' and float(elems[28]) >= 0.5:
            purfilt[elems[3]] = 1

data.close() 

data = None 
lines = None 
line = None 
elems = None 


donidmap = {}

rdfile = open('../../../datasets/PCAWG/donors_and_biospecimens/pcawg_sample_sheet.tsv')
lines = rdfile.readlines() 

for line in lines: 
    if not line.startswith('donor'):
        elems = line.split()
        if elems[3] in purfilt.keys():
            donidmap[elems[2]] = elems[3] 


rdfile.close() 

purfilt = None 

elems = None
line = None 
lines = None 
rdfile = None


medmtcn = {}
canclist = []
cnlist = []


## median mtDNA-CN from samples for which we have mutation data as well 

mtdata = open('../results_June2025/CancerWise_median_mtDNA-CN.txt')
lines = mtdata.readlines()

for line in lines: 
    if line.startswith('Biliary'):
        elems = line.split()
        no = len(elems)
        
        for i in range(0,no): 
            canc = elems[i]
            canc = re.sub(r'\.','-',canc)
            canclist.append(canc)
            canc = None 
    if not line.startswith('Biliary'):
        elems = line.split()
        no = len(elems) 
        for i in range(0,no):
            cnlist.append(elems[i])

for i in range(0,no):
    medmtcn[canclist[i]] = cnlist[i]
    #print(canclist[i],cnlist[i])


mtdata.close() 

i = None 
elems = None 
lines = None 
mtdata = None 
canclist = None 
cnlist = None 
no = None 



## median mtDNA-CN calculated from TCMA database - only for samples for which gene expression data is available 


rdfile = open('../geneExpression_data/transcript/Processed_pcawg.rnaseq.transcript.expr.counts.tsv')
lines = rdfile.readlines()

inclist = {}
inclist2 = {}
duplist = {}

for line in lines:
    line = line.rstrip('\t')
    
    if line.startswith('Transcript'):
        elems = line.split()
        no = len(elems)
        for i in range(1,no):
            sname = elems[i].split('.')
            #print(sname[0],sname[1])
            if not 'Normal' in sname[1] and 'Primary' in sname[1]:
                inclist[sname[0]] = 1 
                if not sname[0] in duplist:
                    inclist2[elems[i]] = 1
                    duplist[sname[0]] = sname[1]
                if sname[0] in duplist: 
                    if 'Primary' in duplist[sname[0]] and 'Primary' in sname[1]:
                        if 'other' in sname[1] and 'solid_tissue' in sname[1]:
                            inclist2[elems[i]] = 1
                            duplist[sname[0]] = sname[1]
                    if 'Metastatic' in duplist[sname[0]] and 'Primary' in sname[1]: 
                        inclist2[elems[i]] = 1
                        duplist[sname[0]] = sname[1]
                    #if 'Metastatic' in duplist[sname[0]] and 'Recurrent' in sname[1]: 
                    #    inclist2[elems[i]] = 1
                    #    duplist[sname[0]] = sname[1]    
                    if 'Recurrent' in duplist[sname[0]] and 'Primary' in sname[1]: 
                        inclist2[elems[i]] = 1
                        duplist[sname[0]] = sname[1]                     
            sname = None 
        elems = None 
        no = None 
        break 
        
    line = None 

lines = None 

rdfile.close()
rdfile = None 
duplist = None 

#print(inclist2)
#break

print ("\nmtDNA CN calculated from TCMA data\n")

medmtcn2 = {}
idcancmap = {}
mtcnlist = {}

mtdata = open('../../../datasets/TCMA_CNA/TCMA-CopyNumber.tsv')
lines = mtdata.readlines()

for line in lines: 
    if not line.startswith('sample_id'):
        elems = line.split()
        #print(elems[2])
        if elems[0] in donidmap.keys():
            donid = donidmap[elems[0]]

            if donid in inclist.keys(): 
                idcancmap[donid] = elems[1]
                mtcnlist[donid] = elems[2]
                
                if elems[1] in medmtcn2.keys():
                    val = medmtcn2[elems[1]]
                    val.append(float(elems[2]))
                    medmtcn2[elems[1]] = val
                else:
                    medmtcn2[elems[1]] = [float(elems[2])]
           
            donid = None
            
        elems = None 
        

sg = dict(sorted(medmtcn2.items(),reverse=True))

medlist_final = {}

for key in sg.keys():
    median = statistics.median(medmtcn2[key])
    print(key,round(median,2))
    medlist_final[key] = median 
    median = None 
    
mtdata.close()

val = None 
elems = None 
lines = None 
mtdata = None 
donidmap = None 
medmtcn = None 
medmtcn2 = None     


    
## Sample classification according to mtDNA-CN 
    
rdfile = open('../geneExpression_data/transcript/Processed_pcawg.rnaseq.transcript.expr.counts.tsv')

wrfile1 = open('../geneExpression_data/transcript/Processed3_pcawg.rnaseq.transcript.expr.counts.csv','w')
wrfile2 = open('../geneExpression_data/transcript/DonorID_canctype_mtDNA-CN.csv','w')
wrfile2.write('DonorID,Cancer_type,mtDNA-CN_class,mtDNA-CN,median_mtDNA-CN,norm_mtDNA-CN\n')

lines = rdfile.readlines()

collist = []
ln = -1

for line in lines:
    line = line.rstrip('\t')
    if line.startswith('Transcript'):
        elems = line.split()
        no = len(elems)
        wrfile1.write('Transcript')
        for i in range(1,no):
            sname = elems[i].split('.')
            if elems[i] in inclist2.keys():
                donid = sname[0]
                #print(donid,sname[1])
                if donid in idcancmap.keys():
                    canctype = idcancmap[donid]
                    ratio = round((float(mtcnlist[donid])/float(medlist_final[canctype])),2)
                    #print(donid,canctype,round(float(mtcnlist[donid]),2),round(float(medlist_final[canctype]),2),ratio)
                    if ratio < 1:
                        collist.append(i)
                        pid = canctype + '_LowCopy_' + donid 
                        wrfile1.write(',{}'.format(pid))
                        wrfile2.write('{},{},{},{},{},{}\n'.format(pid,canctype,"LowCopy",round(float(mtcnlist[donid]),2),round(float(medlist_final[canctype]),2),ratio)) 
                    if ratio > 1: 
                        collist.append(i)
                        pid = canctype + '_HighCopy_' + donid 
                        wrfile1.write(',{}'.format(pid))
                        wrfile2.write('{},{},{},{},{},{}\n'.format(pid,canctype,"HighCopy",round(float(mtcnlist[donid]),2),round(float(medlist_final[canctype]),2),ratio)) 

                    pid = None 
                    canctype = None 
                    ratio = None 
        wrfile1.write('\n')
        elems = None 
        no = None 
        sname = None 
                    
    if not line.startswith('Transcript'):
        elems = line.split()
        no = len(elems)
        wrfile1.write('{}'.format(elems[0]))
        for i in range(1,no):
            if i in collist:
                wrfile1.write(',{}'.format(round(float(elems[i]))))
        wrfile1.write('\n')
        elems = None 
        no = None 
            
    ln = ln + 1
    if ln==1 or ln%1000 == 0:
        print(ln)
    #ln += 1 
        
rdfile.close() 

wrfile1.close()
wrfile2.close()

inclist2 = None 
ln = None 
mtcnlist = None 
medlist_final = None 
idcancmap = None 
collist = None 
        


mtDNA CN calculated from TCMA data

Uterus-AdenoCA 165.81
Thy-AdenoCA 224.32
Stomach-AdenoCA 312.61
SoftTissue-Liposarc 118.09
SoftTissue-Leiomyo 137.63
Skin-Melanoma 231.8
Prost-AdenoCA 240.09
Panc-AdenoCA 187.23
Ovary-AdenoCA 677.83
Lymph-CLL 181.15
Lymph-BNHL 293.51
Lung-SCC 335.07
Lung-AdenoCA 306.5
Liver-HCC 329.98
Kidney-RCC 286.24
Kidney-ChRCC 540.7
Head-SCC 169.29
Eso-AdenoCA 153.03
ColoRect-AdenoCA 683.83
Cervix-SCC 105.54
Cervix-AdenoCA 153.95
CNS-Oligo 487.81
CNS-GBM 392.67
Breast-LobularCA 193.57
Breast-AdenoCA 191.89
Bladder-TCC 195.81
Biliary-AdenoCA 293.86
0
1
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
50000
51000
52000
53000
54000
55000
56000
57000
58000
59000
60000
61000
62000
63000
64000
65000
66000
67000
68000
69000
70000
71000


In [2]:
## Separation of data according to cancer types 
## Separation of Metadata 

import pandas as pd 

df = pd.read_csv('../geneExpression_data/transcript/Processed3_pcawg.rnaseq.transcript.expr.counts.csv')
meta = pd.read_csv('../geneExpression_data/transcript/DonorID_canctype_mtDNA-CN.csv')
#print(meta)
#df
genelist = df.filter(regex = '^Transcript', axis = 1)
#genelist

canctypelist = ['Biliary-AdenoCA',
                'Bladder-TCC',
                'Bone-Benign',
                'Bone-Epith',
                'Bone-Osteosarc',
                'Breast-AdenoCA',
                'Breast-DCIS',
                'Breast-LobularCA',
                'CNS-Medullo',
                'CNS-GBM',
                'CNS-Oligo',
                'CNS-PiloAstro',
                'Cervix-AdenoCA',
                'Cervix-SCC',
                'ColoRect-AdenoCA',
                'Eso-AdenoCA',
                'Head-SCC',
                'Kidney-ChRCC',
                'Kidney-RCC',
                'Liver-HCC',
                'Lung-AdenoCA',
                'Lung-SCC',
                'Lymph-BNHL',
                'Lymph-CLL',
                'Myeloid-AML',
                'Myeloid-MDS',
                'Myeloid-MPN',
                'Ovary-AdenoCA',
                'Panc-AdenoCA',
                'Panc-Endocrine',
                'Prost-AdenoCA',
                'Skin-Melanoma',
                'SoftTissue-Leiomyo',
                'SoftTissue-Liposarc',
                'Stomach-AdenoCA',
                'Thy-AdenoCA',
                'Uterus-AdenoCA']

wrf = open('../geneExpression_data/transcript/Statistics_All_cancerTypes.txt','w')
wrf.write('CancType\tTotSamplesExpr\tNumLowCopy\tNumHighCopy\n')

for canctype in canctypelist:
    print(canctype) 
    var = canctype + '_df'
    var2 = 'ct_' + var 
    var = df.filter(regex = canctype, axis=1) 
    var = var.reindex(sorted(var.columns, reverse = True), axis=1)
    var2 = pd.concat([genelist, var], axis=1) 
    filename = '../geneExpression_data/transcript/canctype_specific_data/'+ canctype + '_counts_data.txt'
    var2.to_csv(filename, sep=' ', index=False)

    filename = None 
    var2 = None 
    var = None 
    #exit()

    #cns_gbm_df = df.filter(regex='^CNS-GBM',axis=1)
    #CNS_GBM_df

    metavar = 'meta' + '_' + canctype 
    metavar = meta[meta['Cancer_type'] == canctype]
    metavar = metavar.sort_values(['DonorID'], ascending = False)
    metafile = '../geneExpression_data/transcript/canctype_specific_data/Metadata/Metadata_' + canctype + '.txt'
    wrf.write('{}\t{}\t{}\t{}\n'.format(canctype,len(metavar),len(metavar[metavar['mtDNA-CN_class'] == 'LowCopy']),len(metavar[metavar['mtDNA-CN_class'] == 'HighCopy'])))
    metavar.to_csv(metafile,sep=' ', index=False)  

    metafile = None 
    metavar = None 
    #meta_cns_gbm = meta[meta['Cancer_type'] == 'CNS-GBM']
    #print(meta_cns_gbm)

wrf.close() 

meta = None 
df = None 
#%reset -f 

Biliary-AdenoCA
Bladder-TCC
Bone-Benign
Bone-Epith
Bone-Osteosarc
Breast-AdenoCA
Breast-DCIS
Breast-LobularCA
CNS-Medullo
CNS-GBM
CNS-Oligo
CNS-PiloAstro
Cervix-AdenoCA
Cervix-SCC
ColoRect-AdenoCA
Eso-AdenoCA
Head-SCC
Kidney-ChRCC
Kidney-RCC
Liver-HCC
Lung-AdenoCA
Lung-SCC
Lymph-BNHL
Lymph-CLL
Myeloid-AML
Myeloid-MDS
Myeloid-MPN
Ovary-AdenoCA
Panc-AdenoCA
Panc-Endocrine
Prost-AdenoCA
Skin-Melanoma
SoftTissue-Leiomyo
SoftTissue-Liposarc
Stomach-AdenoCA
Thy-AdenoCA
Uterus-AdenoCA
